# 🔍 Simple RAG Demo
**Retrieval-Augmented Generation** — without any API keys!

This notebook shows the core RAG concept:
1. Embed documents into a vector index
2. Embed a query
3. Find the most similar documents
4. Feed them as context to an LLM

> 🦃 *Created by Crunch, managed at [Copilotclaw](https://github.com/Copilotclaw/trainingdemo)*

In [ ]:
# Install dependencies (only needed in Colab / fresh env)
!pip install sentence-transformers faiss-cpu --quiet

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

print('✅ Imports done')

## Step 1 — Build the Knowledge Base
These are the documents our RAG system knows about.

In [ ]:
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines search with LLMs.",
    "FAISS is a library by Meta for efficient similarity search on dense vectors.",
    "Sentence transformers convert text into meaningful numerical embeddings.",
    "Local RAG runs entirely on your machine — no API calls, no data leaving your system.",
    "Ollama lets you run models like Llama 3 or Mistral locally with one command.",
    "Vector databases store embeddings and support fast approximate nearest-neighbor search.",
    "The retrieval step finds the top-k most relevant chunks from your knowledge base.",
    "Chunking splits long documents into smaller pieces before embedding them.",
]

print(f'📚 {len(documents)} documents loaded')

## Step 2 — Embed the Documents

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')  # ~80MB, fast, great quality

embeddings = model.encode(documents, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print(f'✅ Embeddings shape: {embeddings.shape}')  # (8, 384)

## Step 3 — Build the FAISS Index

In [ ]:
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)  # L2 = Euclidean distance
index.add(embeddings)

print(f'🗂️  FAISS index built with {index.ntotal} vectors')

## Step 4 — Query the Index

In [ ]:
def search(query: str, top_k: int = 3) -> list[str]:
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, top_k)
    results = [(documents[i], round(float(distances[0][j]), 4)) for j, i in enumerate(indices[0])]
    return results

# Try it!
query = "How does local RAG work without internet?"
print(f'🔎 Query: "{query}"\n')
for doc, score in search(query):
    print(f'  [{score:.4f}] {doc}')

## Step 5 — Simulate the Generation Step
In a real RAG system, the retrieved chunks go into an LLM prompt.
Here we just format what the prompt would look like.

In [ ]:
def build_prompt(query: str, top_k: int = 3) -> str:
    context_docs = search(query, top_k)
    context = '\n'.join([f'- {doc}' for doc, _ in context_docs])
    return f"""You are a helpful assistant. Use the context below to answer the question.

Context:
{context}

Question: {query}

Answer:"""

print(build_prompt("What is FAISS and why is it useful?"))

## 🏋️ Exercises
1. Add 5 more documents about a topic you know (cooking, music, history)
2. Query with different questions — notice how relevance changes
3. Change `top_k` from 3 to 1 or 5 — how does that affect the context?
4. Try a completely unrelated query — what does FAISS return? Why?

## 🚀 Next Steps (Local RAG)
- Replace the fake `Answer:` prompt with a real Ollama call: `ollama.chat(model='llama3', messages=[...])`
- Load real documents: PDFs, markdown files, code
- Add proper chunking with `langchain.text_splitter`
- Persist the FAISS index to disk with `faiss.write_index()`